In [121]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [122]:
df = pd.read_excel("/content/ASL_alphabets.xlsx", index_col=False)

In [123]:
df = df.drop([0,1],axis=1)
print(len(df))
print(df.head())
# [0,1,2,3,4] -> [A,Y,U,S,H]

In [124]:
df = df.iloc[:,1:]
print(df.head())

In [125]:
from sklearn.decomposition import PCA
X = df
X = X.drop(["class"],axis=1)
X.columns = X.columns.astype(str)
pca = PCA()
pca_data = pca.fit_transform(X)
print(len(pca.explained_variance_ratio_))
plt.bar(range(1,len(pca.explained_variance_)+1),height=pca.explained_variance_ratio_)
plt.xlabel('PCA Feature')
plt.ylabel('Explained variance')
plt.title('Feature Explained Variance')
plt.show()

In [127]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X = df.drop(["class"],axis=1)
y = df["class"]
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
gm = GaussianMixture(n_components=5, random_state=0).fit(X_train)

In [130]:
cluster_means = []
for i in range(0,5):
  cluster_means.append(df.loc[df['class']==i].mean().drop('class').values)
cluster_means = np.array(cluster_means)

In [131]:
probabilities = gm.predict_proba(X_test)
predicted_clusters = np.empty(X_test.shape[0])
for idx, sample in enumerate(X_test.values):
    if np.max(probabilities[idx]) > 0.99:
        distances = np.linalg.norm(cluster_means - sample, axis=1)
        closest_cluster = np.argmin(distances)
        predicted_clusters[idx] = closest_cluster
    else:
        predicted_clusters[idx] = -1
y_test = y_test.reset_index(drop=True)
predicted_clusters = pd.Series(predicted_clusters)
filtered_predictions = predicted_clusters[predicted_clusters != -1]
filtered_y_test = y_test[predicted_clusters != -1]
accuracy = accuracy_score(filtered_y_test, filtered_predictions)
print(f"Accuracy: {accuracy}")

In [136]:
import joblib
gm = GaussianMixture(n_components=5, random_state=0).fit(X)
joblib.dump(gm, 'gmm_model.pkl')

In [141]:
gm_loaded = joblib.load('gmm_model.pkl')
gmm_to_cluster_map = []
for gmm_mean in gm_loaded.means_:
    distances = np.linalg.norm(cluster_means - gmm_mean, axis=1)
    closest_cluster = np.argmin(distances)
    gmm_to_cluster_map.append(closest_cluster)
print("GMM to Cluster Map:", gmm_to_cluster_map)